# Notebook 2 — Feature engineering and labels

**Goal:** turn the clean SWELL minutes into the exact features our desktop agent can also compute,
add the labels, and save model-ready tables for Notebook 3.

All feature code lives in **`shared/features.py`** — the live app uses the same functions. This notebook
only (a) renames SWELL columns to our names, (b) calls those functions, and (c) builds the labels.

**Input:** `ml/data/processed/swell_clean.csv` (from Notebook 1)

**Outputs** (`ml/data/processed/`):

| File | Features |
|---|---|
| `swell_features_raw.csv` | features without normalisation |
| `swell_features_calibration.csv` | z-scored against each person's first 15 active minutes (**main**, matches the live app) |
| `swell_features_neutral.csv` | z-scored against each person's whole neutral block (as literally described in the project plan; optimistic, see section 5) |

In [1]:
# --- Find the project folder so the notebook works from any working directory ---
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the folder that contains shared/features.py."""
    for folder in [start, *start.parents]:
        if (folder / "shared" / "features.py").exists():
            return folder
    raise FileNotFoundError("Could not find the Commit project folder (shared/features.py).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))  # lets us "import shared.features"

RAW_DIR = REPO_ROOT / "ml" / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "ml" / "data" / "processed"
REPORTS_DIR = REPO_ROOT / "ml" / "reports"
MODELS_DIR = REPO_ROOT / "shared" / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Project folder:", REPO_ROOT)

Project folder: /Users/matrix/Desktop/Commit


In [2]:
import numpy as np
import pandas as pd

from shared import features as F

pd.set_option("display.width", 160)

In [3]:
# --- Settings ---

# Working conditions kept for modelling (R = relax/film is dropped: no computer use).
WORK_CONDITIONS = ["N", "I", "T"]

# Condition names used by the models and the app database (see README section 13).
CONDITION_LABELS = {"N": "neutral", "I": "interruptions", "T": "time_pressure"}

# Calibration length: the live app learns a personal baseline from the first 15 active
# minutes after install (README section 8.3, config calibration.active_minutes).
CALIBRATION_ACTIVE_MINUTES = 15

# Rolling windows in minutes (README section 7.3: 3- and 5-minute windows).
ROLLING_WINDOWS = F.DEFAULT_ROLLING_WINDOWS

# Self-rating used for the load label (RSME mental effort).
LOAD_RATING_COLUMN = "MentalEffort"

## 1. Load and keep only N, I, T

In [4]:
clean = pd.read_csv(PROCESSED_DIR / "swell_clean.csv", parse_dates=["timestamp"])
work = clean[clean["Condition"].isin(WORK_CONDITIONS)].copy()
print("Rows in clean file:", len(clean), " N/I/T rows kept:", len(work))
print(work["Condition"].value_counts().to_string())

Rows in clean file: 3106  N/I/T rows kept: 2655
Condition
N    1027
I     964
T     664


## 2. Feature parity: SWELL columns → our feature names

We keep only signals that our desktop agent can also log (README section 8.2–8.3).

| Our feature | SWELL column(s) | Notes |
|---|---|---|
| `keystrokes` | `SnKeyStrokes` | all key events |
| `error_keys` | `SnErrorKeys` | SWELL also counts Ctrl+Z here; we count backspace + delete |
| `shortcut_keys` | `SnShortcutKeys` | SWELL counts specific shortcuts (Ctrl+C/X/V/Z/S/A, Shift+Tab); we count any key with Cmd/Ctrl/Alt held |
| `clicks` | `SnLeftClicked + SnRightClicked` | |
| `scrolls` | `SnWheel` | |
| `mouse_distance` | `SnMouseDistance` | units differ between machines; per-person z-scores remove the scale |
| `app_switches` | `SnAppChange` | |

**Not used** (our agent does not log them): `SnMouseAct`, `SnDoubleClicked`, `SnDragged`, `SnChars`,
`SnSpecialKeys`, `SnDirectionKeys`, `SnSpaces`, `SnTabfocusChange`.
Our agent's `pause_count`, `pause_total_seconds` and `active_seconds` have no SWELL equivalent, so the model
cannot use them either (the app still logs them for the activity mode).

In [5]:
def swell_to_our_counts(table: pd.DataFrame) -> pd.DataFrame:
    """
    Rename SWELL columns to our raw count names (the feature-parity mapping above).

    In:  clean SWELL rows (SWELL column names).
    Out: a new table with id/label columns and our RAW_COUNT_COLUMNS.
    Why: after this step, SWELL data looks exactly like what our agent logs.
    """
    counts = pd.DataFrame({
        "participant": table["PP"],
        "block": table["Blok"],
        "condition_code": table["Condition"],
        "timestamp": table["timestamp"],
        "keystrokes": table["SnKeyStrokes"],
        "error_keys": table["SnErrorKeys"],
        "shortcut_keys": table["SnShortcutKeys"],
        "clicks": table["SnLeftClicked"] + table["SnRightClicked"],
        "scrolls": table["SnWheel"],
        "mouse_distance": table["SnMouseDistance"],
        "app_switches": table["SnAppChange"],
        "MentalEffort": table["MentalEffort"],
        "Stress": table["Stress"],
    })
    # One continuous recording = one participant's block. Rolling windows stay inside it.
    counts["recording"] = counts["participant"] + "_block" + counts["block"].astype(str)
    return counts.reset_index(drop=True)


counts = swell_to_our_counts(work)
assert list(counts.columns[4:11]) == F.RAW_COUNT_COLUMNS
counts.head()

,participant,block,condition_code,timestamp,keystrokes,error_keys,shortcut_keys,clicks,scrolls,mouse_distance,app_switches,MentalEffort,Stress,recording
0,PP1,1,N,2012-09-18 13:26:00,0,0,0,0,0,0.000000,0,5.1,4.1,PP1_block1
1,PP1,1,N,2012-09-18 13:27:00,0,0,0,0,0,0.000000,0,5.1,4.1,PP1_block1
2,PP1,1,N,2012-09-18 13:28:00,0,0,0,5,1,5691.235163,5,5.1,4.1,PP1_block1
3,PP1,1,N,2012-09-18 13:29:00,0,0,0,7,4,7726.999223,8,5.1,4.1,PP1_block1
4,PP1,1,N,2012-09-18 13:30:00,53,8,0,5,3,6307.768271,1,5.1,4.1,PP1_block1


## 3. Build the features with `shared/features.py`

* ratio features (error-key ratio, shortcut ratio, keyboard share)
* idle flag (no keyboard/mouse input in the minute)
* 3- and 5-minute trailing rolling mean and standard deviation, plus share of idle minutes.
  Windows are **time-based** and stay inside one recording, so the 2 minutes removed in Notebook 1 do not glue unrelated minutes together.

In [6]:
feature_table = F.build_features(counts, time_column="timestamp", group_column="recording", windows=ROLLING_WINDOWS)
FEATURE_NAMES = F.model_feature_names(ROLLING_WINDOWS)

print("Number of model features:", len(FEATURE_NAMES))
print("Idle minutes (no input):", int(feature_table[F.IDLE_COLUMN].sum()))
feature_table[F.PER_MINUTE_FEATURES].describe().round(2)

Number of model features: 52
Idle minutes (no input): 37


,keystrokes,error_keys,shortcut_keys,clicks,scrolls,mouse_distance,app_switches,error_key_ratio,shortcut_ratio,keyboard_share
count,2655.00,2655.00,2655.00,2655.00,2655.00,2655.00,2655.00,2655.00,2655.00,2655.00
mean,85.13,9.27,0.77,7.30,3.99,10435.48,3.62,0.10,0.04,0.72
std,73.63,12.45,1.88,6.81,6.45,11772.21,3.81,0.13,0.14,0.33
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,23.00,1.00,0.00,2.00,0.00,1798.99,0.00,0.01,0.00,0.56
50%,69.00,5.00,0.00,6.00,1.00,6949.83,3.00,0.07,0.00,0.87
75%,131.00,13.00,1.00,11.00,6.00,15097.91,6.00,0.14,0.01,0.97
max,382.00,95.00,22.00,38.00,80.00,81098.58,23.00,1.00,1.00,1.00


## 4. Labels

* `condition`: neutral / interruptions / time_pressure (3 classes)
* `stressed`: 1 for interruptions or time pressure, 0 for neutral
* `load_person_median`: **high** if the block's MentalEffort rating is above that participant's own median rating, else **low**
* `load_overall_median`: **high** if the rating is above the median of all 75 block ratings, else **low**

**Important:** every participant gave only **one rating per condition**, so the load label is constant inside a block.
There are really only 75 labelled blocks behind the ~2,300 minutes. With 3 ratings per person, the per-person
median split marks at most one block per person as "high" (none, if the top two ratings tie).

In [7]:
def add_condition_labels(table: pd.DataFrame) -> pd.DataFrame:
    """
    Add the 3-class `condition` label and the binary `stressed` label.

    In:  feature table with condition_code (N/I/T).
    Out: a copy with the two label columns.
    """
    result = table.copy()
    result["condition"] = result["condition_code"].map(CONDITION_LABELS)
    result["stressed"] = (result["condition_code"] != "N").astype(int)
    return result


def make_load_labels(table: pd.DataFrame) -> pd.DataFrame:
    """
    Build the low/high load labels from the self-rated mental effort, once per block.

    In:  feature table with participant, block and the rating column.
    Out: one row per (participant, block) with the rating and both load labels.
    Why: the rating exists once per block, so the split must be done on blocks, not minutes
         (otherwise long blocks would count more than short ones).
    """
    blocks = table.drop_duplicates(["participant", "block"])[["participant", "block", "condition_code", LOAD_RATING_COLUMN]].copy()
    rating = blocks[LOAD_RATING_COLUMN]

    # Per-person split: compare each block with that person's own median rating.
    person_median = blocks.groupby("participant")[LOAD_RATING_COLUMN].transform("median")
    blocks["load_person_median"] = np.where(rating > person_median, "high", "low")

    # Overall split: compare with the median of all blocks.
    overall_median = rating.median()
    blocks["load_overall_median"] = np.where(rating > overall_median, "high", "low")
    print(f"Overall median MentalEffort: {overall_median}")
    return blocks


labelled = add_condition_labels(feature_table)
load_blocks = make_load_labels(labelled)
labelled = labelled.merge(load_blocks[["participant", "block", "load_person_median", "load_overall_median"]],
                          on=["participant", "block"], how="left", validate="many_to_one")

print("\nBlocks per load label:")
print(load_blocks[["load_person_median", "load_overall_median"]].apply(pd.Series.value_counts).to_string())
people_without_high = (load_blocks.groupby("participant")["load_person_median"].apply(lambda s: (s == "high").sum()) == 0).sum()
print("\nParticipants with no 'high' block under the per-person split (tied top ratings):", people_without_high)

Overall median MentalEffort: 5.7

Blocks per load label:
      load_person_median  load_overall_median
low                   52                   38
high                  23                   37

Participants with no 'high' block under the per-person split (tied top ratings): 2


In [8]:
# How often does the load label agree with the condition? (blocks, not minutes)
print("Per-person load label vs condition (blocks):")
print(pd.crosstab(load_blocks["condition_code"], load_blocks["load_person_median"]).to_string())
print("\nOverall-median load label vs condition (blocks):")
print(pd.crosstab(load_blocks["condition_code"], load_blocks["load_overall_median"]).to_string())
load_blocks.to_csv(REPORTS_DIR / "load_labels_per_block.csv", index=False)

Per-person load label vs condition (blocks):
load_person_median  high  low
condition_code               
I                     10   15
N                      3   22
T                     10   15

Overall-median load label vs condition (blocks):
load_overall_median  high  low
condition_code                
I                      13   12
N                      10   15
T                      14   11


## 5. Per-person normalisation

Each person's features are z-scored against a personal baseline: `z = (value - baseline mean) / baseline std`.

Two ways to choose the baseline minutes:

1. **Calibration (main).** The person's **first 15 active minutes**, exactly like the live app's calibration period.
   In SWELL, block 1 is always the neutral condition, so these minutes come from the start of the neutral block,
   without needing to know any label. These calibration minutes are then **left out of training and testing**
   (the app makes no predictions during calibration either); otherwise they would be near 0 by construction.
2. **Whole neutral block** (the literal wording of README 7.3 step 5). This uses the test person's neutral
   label to build their baseline, so every neutral minute is centred on 0 by construction. That leaks label
   information into the features and over-states accuracy. It is kept **only as an ablation** in Notebook 3.

Minutes with no input at all (idle) are also left out of training/testing, because the app only predicts in active minutes.

In [9]:
# Calibration minutes: each participant's first 15 non-idle minutes.
labelled["is_calibration"] = F.select_calibration_minutes(
    labelled, time_column="timestamp", active_minutes=CALIBRATION_ACTIVE_MINUTES, person_column="participant"
)
print("Calibration minutes per condition (should all be neutral):")
print(labelled.loc[labelled["is_calibration"], "condition"].value_counts().to_string())
assert set(labelled.loc[labelled["is_calibration"], "condition"]) == {"neutral"}

# Rows that models are trained and tested on.
labelled["in_evaluation"] = (labelled[F.IDLE_COLUMN] == 0) & ~labelled["is_calibration"]
print("\nRows used for modelling:", int(labelled["in_evaluation"].sum()), "of", len(labelled))
print(labelled.loc[labelled["in_evaluation"], "condition"].value_counts().to_string())

Calibration minutes per condition (should all be neutral):
condition
neutral    375

Rows used for modelling: 2243 of 2655
condition
interruptions    951
time_pressure    662
neutral          630


In [10]:
# Variant 1: calibration baseline (main).
features_calibration, calibration_baselines = F.normalise_per_person(
    labelled, "participant", labelled["is_calibration"], FEATURE_NAMES
)

# Variant 2: whole neutral block (active minutes) as the baseline (ablation only).
is_neutral_active = (labelled["condition"] == "neutral") & (labelled[F.IDLE_COLUMN] == 0)
features_neutral, _ = F.normalise_per_person(labelled, "participant", is_neutral_active, FEATURE_NAMES)

# Quick look: mean z-score of a few features per condition (calibration variant, evaluation rows).
check_columns = ["keystrokes", "app_switches", "error_key_ratio", "keystrokes_mean_5m"]
evaluation_rows = features_calibration[features_calibration["in_evaluation"]]
evaluation_rows.groupby("condition")[check_columns].mean().round(2)

,keystrokes,app_switches,error_key_ratio,keystrokes_mean_5m
condition,,,,
interruptions,0.38,1.12,0.16,1.03
neutral,0.38,0.45,0.27,0.89
time_pressure,0.79,0.72,0.29,1.76


## 6. Save

In [11]:
# Columns written to every file: ids, labels, flags, then the features in model order.
ID_AND_LABEL_COLUMNS = [
    "participant", "block", "recording", "timestamp", "condition_code", "condition", "stressed",
    "MentalEffort", "Stress", "load_person_median", "load_overall_median",
    F.IDLE_COLUMN, "is_calibration", "in_evaluation",
]
OUTPUT_COLUMNS = ID_AND_LABEL_COLUMNS + FEATURE_NAMES

outputs = {
    "swell_features_raw.csv": labelled,
    "swell_features_calibration.csv": features_calibration,
    "swell_features_neutral.csv": features_neutral,
}
for file_name, table in outputs.items():
    table[OUTPUT_COLUMNS].to_csv(PROCESSED_DIR / file_name, index=False)
    print("Saved", file_name, table[OUTPUT_COLUMNS].shape)

Saved swell_features_raw.csv (2655, 66)
Saved swell_features_calibration.csv (2655, 66)


Saved swell_features_neutral.csv (2655, 66)


## Summary

* SWELL columns were mapped to the 7 raw counts our agent logs; 8 SWELL columns were dropped for parity.
* `shared/features.py` built 52 features: 10 per-minute features, plus for each of the 3- and 5-minute windows a rolling mean and std of each, and the idle share.
* Labels: 3-class `condition`, binary `stressed`, and two versions of the low/high `load` label from self-rated mental effort.
* The main features are z-scored against each person's first 15 active minutes, like the app's calibration. Those minutes and idle minutes are excluded from modelling.